# Clickbait: PhoBERT level (Kaggle GPU)
Fine-tune `vinai/phobert-base-v2` on ViClickbait-2025 and score the VnExpress dev headlines.

Runs (switch expensive parts off in the configuration cell):
- **Segmentation ablation:** title only, word segmentation (pyvi) on vs off, 3 seeds (42, 43, 44); dev score = mean over seeds.
- **Input ablation** (`RUN_SAPO`): title + sapo (sentence pair, max 160 tokens).
- **Threshold:** besides 0.5, an F1-optimal threshold chosen on a stratified 10% hold-out of ViClickbait
  (never on dev). All ViClickbait-trained models train on the other 90%.
- **Training data ablation** (`RUN_OOF`): ViClickbait (90%) + dev, 5-fold out-of-fold over dev (seed 42 only).
- **Final model** (`RUN_FINAL`): best config retrained on ViClickbait (90%) + all dev, scores `corpus_headlines.csv`.

Input: private Kaggle dataset built with `python -m src.classifiers.clickbait kaggle-bundle`
(`viclickbait.csv`, `dev_headlines.csv`, `corpus_headlines.csv`). Settings: GPU T4 x2 (one GPU is used) or P100, Internet on.

Output in /kaggle/working (download into `results/clickbait/kaggle/`):
`dev_scores_<config>.csv` (mean over seeds, pred at 0.5), `dev_scores_<config>_tuned.csv` (pred at the hold-out threshold),
`dev_scores_<config>_seed<s>.csv`, `holdout_scores_<config>.csv`, `dev_scores_phobert_vc_plus_dev_oof[_tuned].csv`,
`corpus_scores_phobert.csv` (+ `corpus_scores_phobert_tuned.csv`), `phobert_runs.json`, and the folder `phobert_clickbait_final/` (safetensors).

In [ ]:
!pip install -q pyvi

In [ ]:
# ---- Configuration: switch expensive parts off here ----
RUN_SAPO = True              # title + sapo ablation (about 3 runs x 2-3 min on T4)
SAPO_SEGMENTATION = "best"   # "best": only the segmentation setting that wins on title; "both": seg and noseg
RUN_OOF = True               # ViClickbait + dev, 5-fold out-of-fold over dev, seed 42 only (5 runs)
RUN_FINAL = True             # retrain FINAL_CONFIG on ViClickbait + all dev, score corpus_headlines.csv
FINAL_CONFIG = None          # e.g. "phobert_seg"; None = best mean dev PR-AUC among the configs run below
                             # (also the config used for the out-of-fold run)

MODEL = "vinai/phobert-base-v2"
SEEDS = [42, 43, 44]
OOF_SEED, FINAL_SEED = 42, 42
EPOCHS, LR, BATCH, WARMUP, WEIGHT_DECAY = 4, 2e-5, 32, 0.1, 0.01
MAX_LEN_TITLE, MAX_LEN_SAPO = 64, 160
HOLDOUT_FRAC = 0.10          # stratified hold-out of ViClickbait for the threshold (random_state 42)
FP16 = True                  # mixed precision on GPU
TUNED = "_tuned"             # file suffix of the threshold-tuned variant (read by `collect`)

In [ ]:
import gc, glob, json, os, random, time
import numpy as np, pandas as pd, torch, transformers
from pyvi import ViTokenizer
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          get_linear_schedule_with_warmup)


def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    return hits[0] if hits else None


assert find("viclickbait.csv"), "Add the Kaggle dataset built by `kaggle-bundle` (Add Input)"
OUT = "/kaggle/working"
device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = FP16 and device == "cuda"


def read(name):
    df = pd.read_csv(find(name), dtype={"id": str})
    df["title"] = df["title"].fillna("").astype(str)
    df["sapo"] = df["sapo"].fillna("").astype(str)
    return df


vc = read("viclickbait.csv")
dev = read("dev_headlines.csv")
corpus = read("corpus_headlines.csv") if find("corpus_headlines.csv") else None
tr_idx, ho_idx = train_test_split(np.arange(len(vc)), test_size=HOLDOUT_FRAC, stratify=vc["y"], random_state=42)
vc_train = vc.iloc[tr_idx].reset_index(drop=True)
vc_hold = vc.iloc[ho_idx].reset_index(drop=True)

ENV = {"gpu": torch.cuda.get_device_name(0) if device == "cuda" else "cpu",
       "gpu_count": torch.cuda.device_count(), "torch": torch.__version__, "cuda": torch.version.cuda,
       "transformers": transformers.__version__}
LOG = {"env": ENV, "model": MODEL,
       "hyperparameters": {"epochs": EPOCHS, "lr": LR, "batch": BATCH, "warmup": WARMUP, "weight_decay": WEIGHT_DECAY,
                           "max_len_title": MAX_LEN_TITLE, "max_len_title_sapo": MAX_LEN_SAPO, "fp16": use_amp,
                           "seeds": SEEDS, "grad_clip": 1.0},
       "data": {"viclickbait": len(vc), "viclickbait_train": len(vc_train), "viclickbait_holdout": len(vc_hold),
                "holdout_rule": f"stratified {HOLDOUT_FRAC:.0%} of ViClickbait, random_state 42; thresholds chosen here, never on dev",
                "dev": len(dev), "dev_positive": int(dev["y"].sum()),
                "corpus": None if corpus is None else len(corpus)},
       "runs": [], "configs": {}, "oof": None, "final": None}


def save_log():
    with open(f"{OUT}/phobert_runs.json", "w") as f:
        json.dump(LOG, f, indent=2, ensure_ascii=False, default=float)


print(ENV)
print(f"ViClickbait {len(vc)} (train {len(vc_train)}, hold-out {len(vc_hold)}), positive share {vc['y'].mean():.3f}")
print(f"dev {len(dev)}, positive share {dev['y'].mean():.3f}; corpus {None if corpus is None else len(corpus)}")
if corpus is None:
    print("WARNING: corpus_headlines.csv not found; the final model will not score the corpus")

In [ ]:
_seg_cache = {}


def seg(text):
    if text not in _seg_cache:
        _seg_cache[text] = ViTokenizer.tokenize(text) if text.strip() else text
    return _seg_cache[text]


def encode(tok, df, segment, sapo, with_labels=False):
    f = seg if segment else (lambda t: t)
    titles = [f(t) for t in df["title"]]
    sapos = [f(s) for s in df["sapo"]] if sapo else None
    enc = tok(titles, sapos, truncation=True, max_length=MAX_LEN_SAPO if sapo else MAX_LEN_TITLE)
    items = [{k: enc[k][i] for k in enc.keys()} for i in range(len(titles))]
    if with_labels:
        for it, y in zip(items, df["y"]):
            it["labels"] = int(y)
    return items


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def cleanup():
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()


def make_scaler():
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        return torch.amp.GradScaler("cuda", enabled=use_amp)
    return torch.cuda.amp.GradScaler(enabled=use_amp)


def train_model(df, segment, sapo, seed):
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).to(device)
    g = torch.Generator(); g.manual_seed(seed)
    dl = DataLoader(encode(tok, df, segment, sapo, with_labels=True), batch_size=BATCH, shuffle=True,
                    collate_fn=DataCollatorWithPadding(tok), generator=g)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    steps = EPOCHS * len(dl)
    sch = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
    scaler = make_scaler()
    model.train()
    for _ in range(EPOCHS):
        for b in dl:
            b = {k: v.to(device) for k, v in b.items()}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                loss = model(**b).loss
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad(set_to_none=True)
    model.eval()
    return model, tok


@torch.no_grad()
def predict(model, tok, df, segment, sapo):
    dl = DataLoader(encode(tok, df, segment, sapo), batch_size=128, collate_fn=DataCollatorWithPadding(tok))
    out = []
    for b in dl:
        b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(**b).logits
        out.append(torch.softmax(logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out)


def best_threshold(y, s):
    """F1-optimal threshold on (y, s); used only on the ViClickbait hold-out."""
    p, r, t = precision_recall_curve(y, s)
    f1 = 2 * p[:-1] * r[:-1] / np.clip(p[:-1] + r[:-1], 1e-12, None)
    return float(t[int(np.argmax(f1))])


def metrics(y, s, thr=0.5):
    pred = (s >= thr).astype(int)
    return {"f1": float(f1_score(y, pred, zero_division=0)), "pr_auc": float(average_precision_score(y, s)),
            "predicted_positive_share": float(pred.mean())}


def write_scores(name, ids, s, thr=0.5, prefix="dev_scores"):
    pd.DataFrame({"id": ids, "score": np.round(s, 6), "pred": (s >= thr).astype(int)}).to_csv(
        f"{OUT}/{prefix}_{name}.csv", index=False)


CONFIGS = {}  # name -> (segment, sapo)


def run_config(name, segment, sapo):
    CONFIGS[name] = (segment, sapo)
    dev_s, hold_s, t_cfg = [], [], time.time()
    for seed in SEEDS:
        t0 = time.time()
        model, tok = train_model(vc_train, segment, sapo, seed)
        s, h = predict(model, tok, dev, segment, sapo), predict(model, tok, vc_hold, segment, sapo)
        del model; cleanup()
        thr = best_threshold(vc_hold["y"], h)
        write_scores(f"{name}_seed{seed}", dev["id"], s)
        m = {"config": name, "seed": seed, "segment": segment, "sapo": sapo, "seconds": round(time.time() - t0, 1),
             **metrics(dev["y"], s), "holdout_threshold": thr,
             **{f"tuned_{k}": v for k, v in metrics(dev["y"], s, thr).items()}}
        print({k: round(v, 3) if isinstance(v, float) else v for k, v in m.items()})
        LOG["runs"].append(m); dev_s.append(s); hold_s.append(h)
    s, h = np.mean(dev_s, axis=0), np.mean(hold_s, axis=0)
    thr = best_threshold(vc_hold["y"], h)
    write_scores(name, dev["id"], s)
    write_scores(name + TUNED, dev["id"], s, thr)
    pd.DataFrame({"id": vc_hold["id"], "y": vc_hold["y"], "score": np.round(h, 6)}).to_csv(
        f"{OUT}/holdout_scores_{name}.csv", index=False)
    seed_ap = [r["pr_auc"] for r in LOG["runs"] if r["config"] == name]
    seed_f1 = [r["f1"] for r in LOG["runs"] if r["config"] == name]
    LOG["configs"][name] = {
        "segment": segment, "sapo": sapo, "max_len": MAX_LEN_SAPO if sapo else MAX_LEN_TITLE, "seeds": SEEDS,
        "train": "ViClickbait 90%", "seconds": round(time.time() - t_cfg, 1),
        "pr_auc_seed_mean": float(np.mean(seed_ap)), "pr_auc_seed_std": float(np.std(seed_ap, ddof=1)),
        "f1_seed_mean": float(np.mean(seed_f1)), "f1_seed_std": float(np.std(seed_f1, ddof=1)),
        "holdout_threshold_mean_scores": thr, "mean_score": metrics(dev["y"], s),
        "mean_score_tuned": metrics(dev["y"], s, thr)}
    save_log()
    print(name, LOG["configs"][name])

In [ ]:
# Segmentation ablation, title only (6 runs)
T_START = time.time()
for segment in (True, False):
    run_config("phobert_seg" if segment else "phobert_noseg", segment, sapo=False)
best_seg = LOG["configs"]["phobert_seg"]["pr_auc_seed_mean"] >= LOG["configs"]["phobert_noseg"]["pr_auc_seed_mean"]
print("better segmentation setting on title:", "seg" if best_seg else "noseg")

In [ ]:
# Input ablation: title + sapo
if RUN_SAPO:
    for segment in ([True, False] if SAPO_SEGMENTATION == "both" else [best_seg]):
        run_config(f"phobert_{'seg' if segment else 'noseg'}_sapo", segment, sapo=True)
pd.DataFrame(LOG["runs"]).groupby("config")[["f1", "pr_auc", "tuned_f1", "predicted_positive_share", "seconds"]].agg(["mean", "std"]).round(3)

In [ ]:
# Config used for the out-of-fold run and the final model
CHOSEN = FINAL_CONFIG or max(LOG["configs"], key=lambda n: LOG["configs"][n]["pr_auc_seed_mean"])
assert CHOSEN in CONFIGS, f"FINAL_CONFIG must be one of {list(CONFIGS)}"
LOG["chosen_config"] = {"name": CHOSEN, "rule": "FINAL_CONFIG" if FINAL_CONFIG else "best mean per-seed dev PR-AUC (labels at run time)"}
save_log()
print("chosen config:", CHOSEN, CONFIGS[CHOSEN])

In [ ]:
# Training data ablation: ViClickbait (90%) + dev, each dev fold scored by a model that never saw it
if RUN_OOF:
    segment, sapo = CONFIGS[CHOSEN]
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof, hold_s, t0 = np.zeros(len(dev)), [], time.time()
    for k, (tr, te) in enumerate(skf.split(dev, dev["y"])):
        df = pd.concat([vc_train, dev.iloc[tr]], ignore_index=True)
        model, tok = train_model(df, segment, sapo, OOF_SEED)
        oof[te] = predict(model, tok, dev.iloc[te], segment, sapo)
        hold_s.append(predict(model, tok, vc_hold, segment, sapo))
        del model; cleanup()
        print(f"fold {k}: {time.time() - t0:.0f}s")
    thr = best_threshold(vc_hold["y"], np.mean(hold_s, axis=0))
    write_scores("phobert_vc_plus_dev_oof", dev["id"], oof)
    write_scores("phobert_vc_plus_dev_oof" + TUNED, dev["id"], oof, thr)
    LOG["oof"] = {"config": "phobert_vc_plus_dev_oof", "based_on": CHOSEN, "segment": segment, "sapo": sapo,
                  "train": "ViClickbait 90% + 4/5 of dev", "folds": "StratifiedKFold(5, shuffle=True, random_state=42) over dev",
                  "seeds": [OOF_SEED], "note": "one seed only, to save GPU time",
                  "seconds": round(time.time() - t0, 1), "holdout_threshold": thr,
                  **metrics(dev["y"], oof), **{f"tuned_{k}": v for k, v in metrics(dev["y"], oof, thr).items()}}
    save_log()
    print(LOG["oof"])

In [ ]:
# Final model: CHOSEN config on ViClickbait (90%) + all dev; the 10% hold-out stays out to tune the threshold
if RUN_FINAL:
    segment, sapo = CONFIGS[CHOSEN]
    t0 = time.time()
    model, tok = train_model(pd.concat([vc_train, dev], ignore_index=True), segment, sapo, FINAL_SEED)
    thr = best_threshold(vc_hold["y"], predict(model, tok, vc_hold, segment, sapo))
    final_dir = f"{OUT}/phobert_clickbait_final"
    model.save_pretrained(final_dir, safe_serialization=True)
    tok.save_pretrained(final_dir)
    meta = {"config": CHOSEN, "segment_with_pyvi": segment, "input": "title + sapo (pair)" if sapo else "title",
            "max_len": MAX_LEN_SAPO if sapo else MAX_LEN_TITLE, "seed": FINAL_SEED,
            "train": "ViClickbait 90% + all dev", "holdout_threshold": thr}
    with open(f"{final_dir}/clickbait_meta.json", "w") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)
    if corpus is not None:
        s = predict(model, tok, corpus, segment, sapo)
        out = pd.DataFrame({"id": corpus["id"], "score": np.round(s, 6), "pred": (s >= 0.5).astype(int),
                            "pred_tuned": (s >= thr).astype(int)})
        out.to_csv(f"{OUT}/corpus_scores_phobert.csv", index=False)
        write_scores("phobert" + TUNED, corpus["id"], s, thr, prefix="corpus_scores")
        meta.update({"corpus_n": len(corpus), "corpus_share_at_0.5": float(out["pred"].mean()),
                     "corpus_share_tuned": float(out["pred_tuned"].mean())})
    del model; cleanup()
    meta["seconds"] = round(time.time() - t0, 1)
    LOG["final"] = meta
    save_log()
    print(meta)

In [ ]:
LOG["total_seconds"] = round(time.time() - T_START, 1)
save_log()
print(f"total {LOG['total_seconds'] / 60:.1f} min on {ENV['gpu']}")
print(sorted(f for f in os.listdir(OUT) if f.endswith((".csv", ".json"))))